# QLoRA fine-tune: Qwen3-4B → crossword-generator SLM

Distills the (Claude + verifier + scaffolding) pipeline into one-shot generation. Trains on `data/sft/train.jsonl` (chat: fixed system contract → minimal size-routed user prompt → verified assistant program), **response-only loss**, dev for validation, `eval` held out for the base-vs-tuned test (see `colab_eval.ipynb`).

## 1. Install (pinned, Qwen3-capable snapshot)
Versions are **pinned**, not `>=`, on purpose: the current `trl` (1.x) **removed** `DataCollatorForCompletionOnlyLM` and **renamed** `SFTConfig(max_seq_length=)` -> `max_length=`, which would break cells 6-7 with `-U`. `trl==0.19.1` is the last release that supports **both** Qwen3 (needs `transformers>=4.51`) and the response-only collator this notebook relies on.

In [4]:
!git pull

fatal: not a git repository (or any of the parent directories): .git


In [1]:
!pip install -q -U 'transformers==4.53.*' 'trl==0.19.1' 'peft==0.16.*' 'bitsandbytes==0.46.*' 'accelerate==1.8.*' 'datasets==3.6.*'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.2/376.2 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.3/365.3 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 65.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompat

## 2. Get the data
**Do ONE of these** (the notebook will not invent data):
- **Clone your repo:** set `REPO_URL` below to your GitHub repo. The committed `data/sft/{train,dev}.jsonl` splits are used as-is.
- **Upload:** put `train.jsonl`/`dev.jsonl` in a Colab folder and set `DATA_DIR` to it (leave `REPO_URL` as the placeholder).

The raw per-run outputs (`runs/`) are gitignored, so a fresh clone has none; the committed splits are already merged + upsampled, so we use them directly and only rebuild when `runs/` is present -- we never clobber the committed data with empties.

In [2]:
REPO_URL = "https://github.com/Avaneesh-Ramesh-07/CrosswordSLM"   # <-- REQUIRED unless you set DATA_DIR to an upload
DATA_DIR = None            # <-- set to an uploaded folder to skip the clone
import os

if DATA_DIR is None:
    assert "<REPO>" not in REPO_URL, (
        "Set REPO_URL to your repo, OR set DATA_DIR to a folder containing "
        "train.jsonl/dev.jsonl that you uploaded via the Files panel."
    )
    if not os.path.exists("slm"):
        !git clone -q $REPO_URL slm
    DATA_DIR = "slm/data/sft"
    # committed splits are already merged+upsampled; only rebuild if the
    # (gitignored) raw per-run outputs are present -- never clobber with empties.
    if os.path.exists("slm/runs"):
        !cd slm && python pipeline/merge_dataset.py --upsample 11=3,15=3

for _f in ("train.jsonl", "dev.jsonl"):
    assert os.path.exists(f"{DATA_DIR}/{_f}"), f"missing {_f} in {DATA_DIR}"
print("data dir:", DATA_DIR, os.listdir(DATA_DIR))

data dir: slm/data/sft ['negatives_eval.jsonl', 'train.jsonl', 'dev.jsonl', 'negatives.jsonl', 'eval.jsonl']


## 3. Config

In [ ]:
# Qwen3-4B instruct. Confirm the exact HF id (alts: "Qwen/Qwen3-4B",
# "Qwen/Qwen3-4B-Instruct"). Start from Instruct for fast SFT.
model_name  = "Qwen/Qwen3-4B-Instruct-2507"  # base model to fine-tune FROM
adapter_dir = "qwen3-4b-crossword-qlora"      # OUTPUT: trained LoRA adapter dir (merged -> adapter_dir + "-merged")
output_dir  = "results"                       # trainer checkpoints + logs

# QLoRA / LoRA
lora_r, lora_alpha, lora_dropout = 32, 64, 0.05
# Qwen attention + MLP projections
target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]

# programs are long (a full generator); give the sequence room
max_seq_length = 4096

num_train_epochs = 3
per_device_train_batch_size = 1
per_device_eval_batch_size  = 1
gradient_accumulation_steps = 16   # effective batch ~16
learning_rate = 2e-4
lr_scheduler_type = "cosine"
warmup_ratio = 0.03
weight_decay = 0.0
logging_steps = 10

## 4. Load Qwen3-4B in 4-bit (QLoRA) + LoRA adapters

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training

# T4 (Colab free tier) has no bf16 -> fall back to fp16 automatically.
bf16_ok = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if bf16_ok else torch.float16
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (no GPU!)"
print(f"GPU: {gpu} | bf16 supported: {bf16_ok} -> compute dtype {compute_dtype}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_name, quantization_config=bnb_config, device_map={"": 0},
    torch_dtype=compute_dtype, trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(
    model, use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

peft_config = LoraConfig(
    r=lora_r, lora_alpha=lora_alpha, lora_dropout=lora_dropout,
    target_modules=target_modules, bias="none", task_type="CAUSAL_LM",
)

## 5. Load data + render the Qwen chat template
Each row is `{messages:[system,user,assistant]}`. We render it with the model's own chat template into a `text` field; the response-only collator (next cell) then masks everything up to the assistant turn so loss is computed **only on the program**.

In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files={
    "train": f"{DATA_DIR}/train.jsonl",
    "dev":   f"{DATA_DIR}/dev.jsonl",
}, )

def render(row):
    # add_generation_prompt=False -> include the assistant turn as the target
    return {"text": tokenizer.apply_chat_template(row["messages"], tokenize=False,
                                                   add_generation_prompt=False)}

ds = ds.map(render, remove_columns=[c for c in ds["train"].column_names if c != "text"])
print(ds)
print("\n--- one rendered example (head) ---\n", ds["train"][0]["text"][:600])

## 6. Response-only loss
Qwen renders the assistant turn after `<|im_start|>assistant\n`. Masking up to that marker means gradients flow only through the generated program, not the (fixed) system contract or user prompt.

In [ ]:
from trl import DataCollatorForCompletionOnlyLM
response_template = "<|im_start|>assistant\n"   # Qwen chat-template assistant marker
collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)
# sanity: confirm the marker tokenizes and is found in a sample
assert response_template in ds["train"][0]["text"], "assistant marker not found — check template"

## 7. Train (dev = in-training validation; eval stays untouched)

In [ ]:
from trl import SFTTrainer, SFTConfig

args = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=per_device_eval_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    learning_rate=learning_rate,
    lr_scheduler_type=lr_scheduler_type,
    warmup_ratio=warmup_ratio,
    weight_decay=weight_decay,
    logging_steps=logging_steps,
    bf16=bf16_ok,
    fp16=not bf16_ok,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_length=max_seq_length,   # canonical arg; max_seq_length is deprecated/ignored
    dataset_text_field="text",
    packing=False,   # required for response-only masking
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="tensorboard",
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["dev"],
    processing_class=tokenizer,   # tokenize the `text` field with our configured tokenizer
    peft_config=peft_config,
    data_collator=collator,
)
trainer.train()

## 8. Save LoRA adapter + merged fp16 model

In [ ]:
trainer.model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print("saved LoRA adapter to", adapter_dir)

# merge to a standalone fp16 model for inference / GGUF export
from peft import PeftModel
import torch, gc
del model, trainer; gc.collect(); torch.cuda.empty_cache()
base = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16,
                                            device_map={"": 0}, trust_remote_code=True)
merged = PeftModel.from_pretrained(base, adapter_dir).merge_and_unload()
merged.save_pretrained(adapter_dir + "-merged")
tokenizer.save_pretrained(adapter_dir + "-merged")
print("saved merged model to", adapter_dir + "-merged")

## 9. Persist to Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/slm_ckpt
# {adapter_dir} is interpolated by IPython from the Python namespace at runtime
!cp -r {adapter_dir} {adapter_dir}-merged /content/drive/MyDrive/slm_ckpt/ 2>/dev/null; echo saved

## Next
Run **`colab_eval.ipynb`** to serve this merged model and score it on the held-out `eval.jsonl` through our sandbox+scorer — the tuned side of the base-vs-tuned table in `GAP_ANALYSIS.md` (unaugmented Opus is ~5–7% valid; target is high pass@1).